# Challenge Three: Developing Multi-Agent Systems

```
Nathan Verrill
student_04_a8b20623816c_Nathan_Verrill
June 2026
```

This notebook demonstrates how to design and build a multi-agent hierarchy using the Google Agent Development Kit (ADK) framework. It establishes a main coordinating root agent that delegates complex user prompts across distinct, specialized sub-agents[cite: 427, 429, 437].

### Cell 1: Environment Setup & Package Installations

In [ ]:
import os
import getpass

# Install the core Google Agent Development Kit framework package
!pip install -q google-adk requests

# Configure baseline target environment parameters
os.environ["GOOGLE_CLOUD_PROJECT"] = "local-agent-development"

# Securely prompt and set the API Key environment variable
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Cloud / AI Studio API Key: ")

print("\n✅ Environment successfully configured.")

### Cell 2: Define Custom Weather and Google Maps Geocoding Tools

In [ ]:
import os
import requests
from typing import Dict, Any

def get_nws_weather(latitude: float, longitude: float) -> Dict[str, Any]:
    """Retrieves the current weather forecast from the National Weather Service API.

    Args:
        latitude (float): Latitude of the location (e.g., 40.7128).
        longitude (float): Longitude of the location (e.g., -74.0060).

    Returns:
        Dict[str, Any]: A dictionary containing the weather periods or forecast metrics.
    """
    headers = {"User-Agent": "LocalMultiAgent/1.0 (agent-development@local.internal)"}
    try:
        points_url = f"https://api.weather.gov/points/{latitude:.4f},{longitude:.4f}"
        res = requests.get(points_url, headers=headers)
        if res.status_code != 200:
            return {"error": f"NWS Points API returned status code {res.status_code}"}
        
        forecast_url = res.json()["properties"]["forecast"]
        forecast_res = requests.get(forecast_url, headers=headers)
        if forecast_res.status_code != 200:
            return {"error": f"NWS Forecast API returned status code {forecast_res.status_code}"}
            
        return {"forecast": forecast_res.json()["properties"]["periods"][0]}
    except Exception as e:
        return {"error": str(e)}


def geocode_place(address: str) -> Dict[str, Any]:
    """Converts a descriptive city name or address into latitude and longitude coordinates.

    Args:
        address (str): The descriptive address or city name (e.g., "Austin, TX").

    Returns:
        Dict[str, Any]: A dictionary containing 'lat' and 'lng' float values.
    """
    api_key = os.getenv("GOOGLE_API_KEY")
    url = f"https://maps.googleapis.com/maps/api/geocode/json?address={address}&key={api_key}"
    try:
        res = requests.get(url)
        data = res.json()
        if data.get("status") == "OK":
            location = data["results"][0]["geometry"]["location"]
            return {"lat": float(location["lat"]), "lng": float(location["lng"])}
        return {"error": f"Google Maps Geocoding failed with status: {data.get('status')}"}
    except Exception as e:
        return {"error": str(e)}

print("✅ Custom tool functions compiled in memory.")

### Cell 3: Setup Callback Functions, Sub-Agents, & Multi-Agent Root Orchestrator

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search, agent_tool
from typing import Any

# --- Shared Telemetry Loggers ---

def log_user_prompt(callback_context: Any, llm_request: Any) -> None:
    """Extracts and prints the incoming user prompt text for session visibility."""
    try:
        if hasattr(llm_request, "contents") and llm_request.contents:
            last_turn = llm_request.contents[-1]
            if hasattr(last_turn, "parts") and last_turn.parts:
                part = last_turn.parts[0]
                text = part.get("text") if isinstance(part, dict) else getattr(part, "text", None)
                if text:
                    agent_name = getattr(callback_context, "agent_name", "agent")
                    print(f"\n📝 [{agent_name}] USER >> {text.strip()}")
    except Exception:
        pass


def log_model_response(callback_context: Any, llm_response: Any) -> None:
    """Extracts and prints the final model response content text output."""
    try:
        if hasattr(llm_response, "content") and llm_response.content:
            content = llm_response.content
            if hasattr(content, "parts") and content.parts:
                part = content.parts[0]
                text = part.get("text") if isinstance(part, dict) else getattr(part, "text", None)
                if text:
                    agent_name = getattr(callback_context, "agent_name", "agent")
                    print(f"🤖 [{agent_name}] MODEL >> {text.strip()}\n")
    except Exception:
        pass


def validate_user_input(callback_context: Any, llm_request: Any) -> None:
    """Inspects inputs to enforce region boundaries and malicious instruction filters."""
    try:
        if not hasattr(llm_request, "contents") or not llm_request.contents:
            return
            
        last_turn = llm_request.contents[-1]
        if not hasattr(last_turn, "parts") or not last_turn.parts:
            return
            
        part = last_turn.parts[0]
        user_text = part.get("text") if isinstance(part, dict) else getattr(part, "text", None)
        
        if not user_text or not isinstance(user_text, str):
            return
            
        user_text_upper = user_text.upper()

        # Enforce geographical restriction to United States operations
        non_us_indicators = ["LONDON", "PARIS", "TOKYO", "BERLIN", "FRANCE", "UK", "JAPAN", "CANADA", "MEXICO"]
        if any(indicator in user_text_upper for indicator in non_us_indicators):
            print(f"🛑 [VALIDATION BLOCKED]: Non-US boundary constraint triggered.")
            refusal = "The National Weather Service API only supports locations within the United States."
            if isinstance(part, dict):
                part["text"] = f"Output exactly this message and nothing else: {refusal}"
            else:
                setattr(part, "text", f"Output exactly this message and nothing else: {refusal}")
            return

        # Mitigate malicious prompt injections and directive bypasses
        malicious_keywords = ["IGNORE PREVIOUS DIRECTIONS", "SYSTEM PROMPT", "DELETE", "DROP TABLE", "YOU ARE NOW A"]
        if any(keyword in user_text_upper for keyword in malicious_keywords):
            print(f"🛑 [VALIDATION BLOCKED]: Directorial instructions or safety risk flagged.")
            refusal = "Safety Alert: This request violates our security validation guidelines."
            if isinstance(part, dict):
                part["text"] = f"Output exactly this message and nothing else: {refusal}"
            else:
                setattr(part, "text", f"Output exactly this message and nothing else: {refusal}")
            return
            
    except Exception:
        pass


def chained_before_callback(callback_context: Any, llm_request: Any) -> None:
    """Orchestrates sequential pre-model intercept actions for security and telemetry."""
    validate_user_input(callback_context, llm_request)
    log_user_prompt(callback_context, llm_request)


# --- DECLARE SPECIALIZED SUB-AGENTS ---

# 1. Weather Specialist Sub-Agent (Handles custom Python function tools)
weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.5-flash",
    instruction="You are a weather expert. Find coordinates of places and fetch real-time NWS data.",
    tools=[geocode_place, get_nws_weather],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response
)

# 2. Search Specialist Agent (Isolated with the built-in search grounding tool to prevent cross-contamination)
search_agent = Agent(
    name="search_agent",
    model="gemini-2.5-flash",
    instruction="You are a search expert. Use the built-in google_search tool to find real-time world data updates.",
    tools=[google_search]
)


# --- DECLARE COORDINATING ROOT ORCHESTRATOR AGENT ---

ROOT_AGENT_INSTRUCTIONS = """
You are the main coordinating root agent. You receive user queries and delegate tasks appropriately:
- For current regional conditions, temperatures, forecasts, and city weather lookups, delegate tasks to 'weather_agent'.
- For generic questions, factual world updates, current events, sports headlines, and general information searches, use the 'search_agent' tool.
Combine their outputs to respond back to the user clearly.
"""

root_agent = Agent(
    name="root_coordinator",
    model="gemini-2.5-flash",
    instruction=ROOT_AGENT_INSTRUCTIONS,
    sub_agents=[weather_agent],  
    tools=[agent_tool.AgentTool(agent=search_agent)],  
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response
)

print("✅ Multi-agent hierarchy successfully established with proper tool separation rules.")

### Cell 4: App Verification & Multi-Agent Execution Evaluation

In [ ]:
from vertexai.preview import reasoning_engines

# Bind the root orchestrator agent into the local interactive application host
app = reasoning_engines.AdkApp(agent=root_agent)

# Test matrix exercising both the weather sub-agent and search sub-agent execution paths
test_queries = [
    "What is the weather like in New York, NY right now?",
    "Give me the news highlights in the world of sports."
]

print("🚀 Querying AdkApp natively across evaluation matrix...\n")
user_id = "local-challenge-three-session"

for query in test_queries:
    print(f"=======================================================")
    print(f"👉 Query: '{query}'")
    print(f"=======================================================")
    
    try:
        # Create an isolated user tracking session
        session = app.create_session(user_id=user_id)
        session_id = session.get("session_id") or session.get("id")
        
        # Execute streaming query context and accumulate final answer updates
        response_text = ""
        for event in app.stream_query(
            user_id=user_id,
            session_id=session_id,
            message=query
        ):
            if isinstance(event, dict) and "content" in event and "parts" in event["content"]:
                for part in event["content"]["parts"]:
                    if "text" in part:
                        response_text += part["text"]
                        
        if response_text:
            print(f"[Final Output text]:\n{response_text.strip()}\n")
            
    except Exception as e:
        print(f"⚠️ App Engine Routing Error: {e}\n")

print("🏁 Multi-agent routing loop validation completed successfully.")